In [2]:
import obspy
import torch
import os
from glob import glob

## Download event data

In [ ]:
!wget -q https://github.com/AI4EPS/PhaseNet/releases/download/test_data/test_data.zip -O test_data.zip
!unzip -q -o test_data.zip


## Run PhaseNet-Plus

In [ ]:
mseed_list = glob('test_data/mseed/*.mseed')
with open("mseed_list.txt", "w") as f:
    f.write("\n".join(mseed_list))

In [3]:
ngpu = torch.cuda.device_count()
# base_cmd = "../predict.py --model phasenet_plus --backbone unet --data_list mseed_list.txt --result_path ./results --format=mseed  --batch_size 1 --workers 1 --resume ../phasenet_plus_unet/model_best.pth"
base_cmd = "../predict.py --model phasenet_tf --backbone unet --data_list mseed_list.txt --result_path ./results --format=mseed  --batch_size 1 --workers 1 --resume ../phasenet_tf_unet/model_best.pth"
base_cmd = "../predict.py --model phasenet --backbone unet --data_list mseed_list.txt --result_path ./results --format=mseed  --batch_size 1 --workers 1 --resume ../phasenet_unet/model_best.pth"


plot_figure = True
if plot_figure:
    base_cmd += " --plot_figure"

/global/home/users/zhuwq0/.local/miniconda3/lib/python3.11/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [5]:
print(f"{ngpu = }")
if ngpu == 0:
    cmd = f"python {base_cmd} --device cpu"
elif ngpu == 1:
    cmd = f"python {base_cmd}"
else:
    cmd = f"torchrun --nproc_per_node {ngpu} {base_cmd}"

print(cmd)
os.system(cmd);

ngpu = 0
python ../predict.py --model phasenet --backbone unet --data_list mseed_list.txt --result_path ./results --format=mseed  --batch_size 1 --workers 1 --resume ../phasenet_unet/model_best.pth --plot_figure --device cpu
Not using distributed mode
Namespace(model='phasenet', resume='../phasenet_unet/model_best.pth', backbone='unet', phases=['P', 'S'], device='cpu', workers=1, batch_size=1, use_deterministic_algorithms=False, amp=False, world_size=1, dist_url='env://', data_path='./', data_list='mseed_list.txt', hdf5_file=None, prefix='', format='mseed', dataset='das', result_path='./results', plot_figure=True, min_prob=0.3, add_polarity=False, add_event=False, sampling_rate=100.0, highpass_filter=None, response_path=None, response_xml=None, subdir_level=0, cut_patch=False, nt=20480, nx=5120, resample_time=False, resample_space=False, system=None, location=None, skip_existing=False, distributed=False)
Total samples:  ./.mseed : 16 files
Loading checkpoint: ../phasenet_unet/model_bes

Predicting: 100%|██████████| 16/16 [06:30<00:00, 24.40s/it]
Merging picks_phasenet: 16it [00:00, 363.24it/s]
Merging events_phasenet: 0it [00:00, ?it/s]


Number of picks: 77
Number of P picks: 41
Number of S picks: 36
Number of event x station: 0


## Plot results

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

for i, f in enumerate(glob('results/figures_phasenet_plus/*.png')):
    img = Image.open(f) 
    axes[i].imshow(img)
    axes[i].axis('off')
    if i >= 15:
        break
plt.tight_layout()
plt.show()